# Weather Rain MLOps: Colab launcher

Runtime: GPU (Runtime > Change runtime type > T4 GPU).

1. Upload `weather_rain_mlops.zip` to `MyDrive/weather_rain_mlops/` (or set REPO_SOURCE to "github").
2. Set SECTION and GROUP in the parameters cell, then Runtime > Run all.
3. Section 1 runs the environment check: versions, GPU, .h5 save, and a Keras to ONNX round trip.
4. Section 2 runs the TensorFlow preprocessing tests and writes the augmentation previews. A CPU runtime is enough for Section 2. Sections 3 and later need the T4 GPU.
6. Section 3: set GROUP = "smoke,timing,A,B,C" and COLAB_UNITS_REMAINING to the number in the Resources panel. smoke runs all tests and a 2-epoch run on a tiny subset, timing measures one epoch and stops if the worst case is above 60% of your units, then A (sanity checks), B (vanilla CNN, 5 seeds) and C (R0, 5 seeds) run. A finished run is never repeated, so after a disconnect just run all again.
5. Results are written to Google Drive under `MyDrive/weather_rain_mlops/experiments`. Copy that folder back into the local repo's `experiments/` folder.

No passwords or tokens are used here. The repo comes from a public GitHub clone or from a zip uploaded to Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/weather_rain_mlops"
import os
os.makedirs(f"{DRIVE_ROOT}/colab_env", exist_ok=True)

In [ ]:
!python --version
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!pip list 2>/dev/null | grep -iE "^(tensorflow|tf-keras|tf_keras|keras|tf2onnx|onnx|onnxruntime|numpy|pandas|pillow|matplotlib|scikit-learn|protobuf|fastapi|uvicorn|locust|streamlit|h5py|ml-dtypes|opencv-python) "
!pip list > {DRIVE_ROOT}/colab_env/pip_list_preinstalled.txt
!python --version > {DRIVE_ROOT}/colab_env/python_version.txt
print("Full pip list saved to", f"{DRIVE_ROOT}/colab_env/pip_list_preinstalled.txt")

In [ ]:
SECTION = 3
GROUP = "smoke,timing,A,B,C"
COLAB_UNITS_REMAINING = 74
REPO_SOURCE = "drive_zip"
GITHUB_URL = ""
REPO_ZIP = f"{DRIVE_ROOT}/weather_rain_mlops.zip"
REPO_DIR = "/content/weather_rain_mlops"
EXPERIMENTS_DIR = f"{DRIVE_ROOT}/experiments"

In [ ]:
import shutil, subprocess
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
if REPO_SOURCE == "github":
    subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, REPO_DIR], check=True)
else:
    shutil.unpack_archive(REPO_ZIP, "/content")
print(sorted(os.listdir(REPO_DIR)))

In [ ]:
LOCAL_DATA = "/content/data"
if os.path.exists(LOCAL_DATA):
    shutil.rmtree(LOCAL_DATA)
shutil.copytree(f"{REPO_DIR}/data", LOCAL_DATA, ignore=shutil.ignore_patterns("quarantine", "uploads", "*.db"))
for split in ["train", "valid", "test"]:
    print(split, {c: len(os.listdir(f"{LOCAL_DATA}/{split}/{c}")) for c in sorted(os.listdir(f"{LOCAL_DATA}/{split}"))})

In [ ]:
%cd {REPO_DIR}
!pip freeze | grep -iE "^(tensorflow|tf-keras|keras|protobuf|numpy|ml-dtypes|ml_dtypes|h5py)==" > /content/colab_constraints.txt
!cat /content/colab_constraints.txt
!pip install -q -r requirements-trainer.txt -c /content/colab_constraints.txt 2>&1 | tail -5
!pip list 2>/dev/null | grep -iE "^(tensorflow|keras|tf2onnx|onnx|onnxruntime|numpy|protobuf) "

In [ ]:
os.environ["DATA_DIR"] = LOCAL_DATA
os.environ["EXPERIMENTS_DIR"] = EXPERIMENTS_DIR
os.environ["COLAB_UNITS_REMAINING"] = str(COLAB_UNITS_REMAINING)
os.makedirs(EXPERIMENTS_DIR, exist_ok=True)
if SECTION == 1:
    !python -m src.check_env --profile trainer | tee {DRIVE_ROOT}/colab_env/check_env_colab.json
elif SECTION == 2:
    !python -m pytest -q tests/test_preprocessing.py tests/test_preprocessing_tf.py 2>&1 | tee {EXPERIMENTS_DIR}/colab_section2.log
    !python -m src.eda --augment-preview 2>&1 | tee -a {EXPERIMENTS_DIR}/colab_section2.log
else:
    LOG_TAG = GROUP.replace(",", "_")
    if "smoke" in GROUP:
        tests = subprocess.run(["python", "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
        open(f"{EXPERIMENTS_DIR}/colab_section{SECTION}_tests.log", "w").write(tests.stdout + tests.stderr)
        print(tests.stdout[-3000:])
        if tests.returncode != 0:
            raise SystemExit("Tests failed, see the output above. No experiment was started.")
    !python -m src.experiments --section {SECTION} --group {GROUP} 2>&1 | tee -a {EXPERIMENTS_DIR}/colab_section{SECTION}_{LOG_TAG}.log